# ML-04 — Search Intelligence Data Contract (Lane 1: Ranking Signal Score)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/alinoor4/flyrank-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This notebook defines the formal **Data Contract** for **Lane 1: Ranking Signal Score**, verifies key facts with real DuckDB warehouse queries against a mid-panel month (`month=2026-03`), builds a 5-feature vector with timing guarantees, and demonstrates the deliberate feature leakage trap on real warehouse data.

> Loaded Skills: `skills/writing-data-contracts/SKILL.md` & `skills/flyrank/flyrank-data/SKILL.md` & `skills/querying-big-datasets/SKILL.md`

In [9]:
# 0. Local Environment Setup & Data Access Setup
import os, getpass, duckdb
import pandas as pd
import numpy as np

# Load Skills
def load_skill(path):
    full_path = f'../../skills/{path}' if os.path.exists(f'../../skills/{path}') else f'skills/{path}'
    if os.path.exists(full_path):
        with open(full_path, 'r', encoding='utf-8') as f:
            content = f.read()
        print(f'--- Loaded Skill: {path} ---\n{content[:300]}...\n')
        return content
    else:
        print(f'Skill file not found at {full_path}')
        return ''

contract_skill = load_skill('writing-data-contracts/SKILL.md')
data_skill = load_skill('flyrank/flyrank-data/SKILL.md')

# Hugging Face Access Setup for Local Execution
HF_TOKEN = os.environ.get('HF_TOKEN')
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get('HF_TOKEN')
    except Exception:
        pass

if not HF_TOKEN:
    print("No HF_TOKEN environment variable found. Please paste your Hugging Face READ token below (or press Enter if using a public token):")
    try:
        HF_TOKEN = getpass.getpass("HF READ Token (hf_...): ")
    except Exception:
        HF_TOKEN = ''

# Connect DuckDB to Hugging Face warehouse
con = duckdb.connect()
if HF_TOKEN:
    con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")
    print("[OK] Registered Hugging Face secret with DuckDB.")
else:
    print("[NOTE] No HF_TOKEN provided. Querying public endpoints.")

REL = 'hf://datasets/FlyRank/internship-warehouse'
MID_PANEL_MONTH = f"{REL}/fact_content_daily_performance/month=2026-03/*.parquet"
SAMPLE_TABLE = f"{REL}/fact_content_daily_performance_sample.parquet"
DIM_CONTENT = f"{REL}/dim_content.parquet"
DIM_CLIENTS = f"{REL}/dim_clients.parquet"
FACT_QUERY = f"{REL}/fact_content_query_90d.parquet"

--- Loaded Skill: writing-data-contracts/SKILL.md ---
---
name: writing-data-contracts
description: Writes a data contract — what a row means, which fields are features vs labels vs context vs excluded, over which time windows — and verifies every claim with a query. Use before any feature building or modeling, or when results look wrong and the data d...

--- Loaded Skill: flyrank/flyrank-data/SKILL.md ---
---
name: flyrank-data
description: The FlyRank internship datasets — the 30k-row starter CSV and its gotchas, the ~79M-row warehouse release tables and grains, panel warnings, access, and iteration rules. Load for EVERY task that touches the data. (Project-specific: delete this folder when reusing ...

No HF_TOKEN environment variable found. Please paste your Hugging Face READ token below (or press Enter if using a public token):
[OK] Registered Hugging Face secret with DuckDB.


## Step 1: The Contract (Plain Words — 5 Answers)

1. **Row Meaning (Unit of Analysis):** One row represents the daily search and analytics performance for a single content item (`content_hash_id`) belonging to a specific client (`client_hash_id`) on a single calendar date (`report_date`).
2. **Tables Used:** Primary fact table `fact_content_daily_performance` (partitioned by month, iterating on `month=2026-03`), joined with `dim_content` for article length/type metadata, `dim_clients` for access flags, and `fact_content_query_90d` for search portfolio diversity.
3. **Time Window:** Mid-panel development window: March 2026 (`month=2026-03`). Features are derived from the first half of the month (Days 1–15), while the outcome target is evaluated on the second half of the month (Days 16–31).
4. **Predict / Rank (Label or Proxy):** **Ranking Signal Score** — ranking pages by their organic search growth opportunity in the future window (Days 16–31 click outcome).
5. **Deliberately Excluded:** `client_hash_id` and `content_hash_id` as learning features (kept solely for grouping/splits), future metrics beyond Day 15 decision timestamp, and static client flags without variance.

## Step 2: Prove Three Facts with DuckDB Queries (Mid-Panel Month = 2026-03)

We execute three small SQL queries against `month=2026-03` to prove the contract claims empirically.

In [10]:
# Query 1: Prove the Grain (Zero duplicate rows at (report_date, client_hash_id, content_hash_id))
q1_grain = f"""
    SELECT report_date, client_hash_id, content_hash_id, COUNT(*) AS duplicate_count
    FROM read_parquet('{MID_PANEL_MONTH}')
    GROUP BY report_date, client_hash_id, content_hash_id
    HAVING COUNT(*) > 1
    LIMIT 5
"""
df_grain = con.sql(q1_grain).df()
print(f"--- Fact 1: Grain Probe ---")
print(f"Duplicate rows returned (should be 0): {len(df_grain)}")

--- Fact 1: Grain Probe ---
Duplicate rows returned (should be 0): 0


In [11]:
# Query 2: Prove Row Count & Date Span for Mid-Panel Month (2026-03)
q2_span = f"""
    SELECT
        COUNT(*)                            AS total_row_count,
        MIN(report_date)                    AS min_report_date,
        MAX(report_date)                    AS max_report_date,
        COUNT(DISTINCT client_hash_id)      AS distinct_clients,
        COUNT(DISTINCT content_hash_id)     AS distinct_content_items
    FROM read_parquet('{MID_PANEL_MONTH}')
"""
df_span = con.sql(q2_span).df()
print(f"--- Fact 2: Row Count & Date Span (March 2026) ---")
print(df_span.to_string(index=False))

--- Fact 2: Row Count & Date Span (March 2026) ---
 total_row_count min_report_date max_report_date  distinct_clients  distinct_content_items
         9841378      2026-03-01      2026-03-31                55                  331437


In [12]:
# Query 3: Prove Availability with IS TRUE Filter
# Note: Nullable flags require IS TRUE to avoid dropping NULLs silently
q3_avail = f"""
    SELECT
        COUNT(*)                                                                                AS total_rows,
        COUNT(CASE WHEN ga4_data_available IS TRUE THEN 1 END)                                 AS ga4_true_rows,
        COUNT(CASE WHEN gsc_data_available IS TRUE THEN 1 END)                                 AS gsc_true_rows,
        COUNT(CASE WHEN ga4_data_available IS TRUE AND gsc_data_available IS TRUE THEN 1 END) AS both_true_rows,
        ROUND(COUNT(CASE WHEN ga4_data_available IS TRUE AND gsc_data_available IS TRUE THEN 1 END) * 100.0 / COUNT(*), 2) AS pct_surviving
    FROM read_parquet('{MID_PANEL_MONTH}')
"""
df_avail = con.sql(q3_avail).df()
print(f"--- Fact 3: Data Availability (IS TRUE Filter Probe) ---")
print(df_avail.to_string(index=False))

--- Fact 3: Data Availability (IS TRUE Filter Probe) ---
 total_rows  ga4_true_rows  gsc_true_rows  both_true_rows  pct_surviving
    9841378         413966        3611061          364347            3.7


## Step 3: Build Five Features (Max) with Timing Guarantees

We build a small feature matrix for **Lane 1: Ranking Signal Score** from March 2026. Features are measured strictly during the **Days 1–15 feature window**, predicting outcome in the **Days 16–31 target window**.

| Feature Name | Field / Formula | Knowable at decision moment because... |
|---|---|---|
| `log_imp_prev` | `np.log1p(SUM(gsc_impressions))` in Days 1–15 | Search impressions during Days 1–15 are logged in GSC prior to the Day 15 decision moment. |
| `avg_position_clean` | `AVG(gsc_avg_position)` in Days 1–15 | Search position ranks are recorded by GSC prior to the Day 15 decision moment. |
| `visible_queries` | `content_visible_query_count` | Query portfolio diversity is captured in the prior 90-day search query snapshot. |
| `word_count_log` | `np.log1p(word_count)` | Content word count is static article metadata established at publication time. |
| `historical_ctr` | `clk_feature / (imp_feature + 1e-5)` | Click-Through Rate in Days 1–15 measures user engagement prior to the decision timestamp. |

In [13]:
# Build small 5-feature matrix using temporal DuckDB pushdown query (Days 1–15 features vs Days 16–31 outcome)
feature_query = f"""
    WITH perf_feature AS (
        SELECT
            content_hash_id                             AS content_id,
            ANY_VALUE(client_hash_id)                   AS client_id,
            SUM(gsc_impressions)                        AS imp_feature,
            SUM(gsc_clicks)                             AS clk_feature,
            AVG(CASE WHEN gsc_avg_position > 0 THEN gsc_avg_position END) AS pos_feature
        FROM read_parquet('{MID_PANEL_MONTH}')
        WHERE gsc_data_available IS TRUE
          AND report_date >= '2026-03-01' AND report_date <= '2026-03-15'
        GROUP BY content_hash_id
        HAVING SUM(gsc_impressions) >= 15
    ),
    perf_target AS (
        SELECT
            content_hash_id                             AS content_id,
            SUM(gsc_clicks)                             AS clk_future
        FROM read_parquet('{MID_PANEL_MONTH}')
        WHERE gsc_data_available IS TRUE
          AND report_date >= '2026-03-16' AND report_date <= '2026-03-31'
        GROUP BY content_hash_id
    ),
    dim AS (
        SELECT content_hash_id AS content_id, word_count
        FROM read_parquet('{DIM_CONTENT}')
    ),
    qmix AS (
        SELECT content_hash_id AS content_id, ANY_VALUE(content_visible_query_count) AS visible_queries
        FROM read_parquet('{FACT_QUERY}')
        GROUP BY content_hash_id
    )
    SELECT
        p.content_id,
        p.client_id,
        p.imp_feature,
        p.clk_feature,
        p.pos_feature,
        COALESCE(t.clk_future, 0) AS clk_future,
        d.word_count,
        q.visible_queries
    FROM perf_feature p
    LEFT JOIN perf_target t ON p.content_id = t.content_id
    LEFT JOIN dim d ON p.content_id = d.content_id
    LEFT JOIN qmix q ON p.content_id = q.content_id
    LIMIT 10000
"""

df_raw = con.sql(feature_query).df()

# Create Honest 5-Feature Frame (Days 1–15 signals ONLY)
X_df = pd.DataFrame()
X_df['log_imp_prev'] = np.log1p(df_raw['imp_feature'].fillna(0))
X_df['avg_position_clean'] = df_raw['pos_feature'].fillna(df_raw['pos_feature'].median())
X_df['visible_queries'] = df_raw['visible_queries'].fillna(0)
wc_clean = df_raw['word_count'].fillna(df_raw['word_count'].median())
X_df['word_count_log'] = np.log1p(wc_clean)
X_df['historical_ctr'] = (df_raw['clk_feature'] / (df_raw['imp_feature'] + 1e-5)).clip(0, 1)

print(f"[OK] Built 5-feature matrix shape: {X_df.shape}")
X_df.head()

[OK] Built 5-feature matrix shape: (10000, 5)


,log_imp_prev,avg_position_clean,visible_queries,word_count_log,historical_ctr
0,3.637586,4.472222,1,8.165079,0.000000
1,4.510860,28.613919,1,8.290794,0.000000
2,6.668228,4.153613,4,8.355145,0.001272
3,6.648985,6.898102,3,7.887584,0.000000
4,6.016157,4.966297,5,8.190909,0.007335


## Step 4: The Leakage Trap (Deliberate Leak Experiment)

Per notebook 02 and `skills/hunting-leakage-and-validating/SKILL.md`, we demonstrate how adding a target-derived feature artificially inflates performance scores to near-perfection.

In [14]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, accuracy_score

# Define Target y: Future High Engagement (Binary: 1 if future clicks in Days 16–31 > median, 0 otherwise)
y_target = (df_raw['clk_future'] > df_raw['clk_future'].median()).astype(int)

# Step A: Add ONE label-derived column on purpose (The Trap)
# Here leaky_future_clicks is derived directly from the Days 16–31 target outcome clk_future!
X_leaky = X_df.copy()
X_leaky['leaky_future_clicks'] = df_raw['clk_future'] * 1.05  # Direct future target leak!

X_tr, X_te, y_tr, y_te = train_test_split(X_leaky, y_target, test_size=0.3, random_state=42)
model_leaky = RandomForestClassifier(n_estimators=50, random_state=42).fit(X_tr, y_tr)
score_leaky = roc_auc_score(y_te, model_leaky.predict_proba(X_te)[:, 1])

# Step B: Remove the leaky column and keep the honest number
X_honest = X_df.copy()
X_tr_h, X_te_h, y_tr_h, y_te_h = train_test_split(X_honest, y_target, test_size=0.3, random_state=42)
model_honest = RandomForestClassifier(n_estimators=50, random_state=42).fit(X_tr_h, y_tr_h)
score_honest = roc_auc_score(y_te_h, model_honest.predict_proba(X_te_h)[:, 1])

print(f"--- Leakage Experiment Results ---")
print(f"1. Leaky Model Score (ROC-AUC): {score_leaky:.4f}  <-- Fake perfect score (leaks Days 16-31 outcome)!")
print(f"2. Honest Model Score (ROC-AUC): {score_honest:.4f}  <-- Honest generalization score (uses Days 1-15 signals only).")

--- Leakage Experiment Results ---
1. Leaky Model Score (ROC-AUC): 1.0000  <-- Fake perfect score (leaks Days 16-31 outcome)!
2. Honest Model Score (ROC-AUC): 0.8635  <-- Honest generalization score (uses Days 1-15 signals only).


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.